In [ ]:
from collections import defaultdict
corpus = [
    "best places to visit in india",
    "best places to visit in chennai",
    "best places to visit near me",
    "best places to visit during summer",
    "best places to visit in kerala",
    "best hotels to stay in chennai",
    "best restaurants near me",
    "places to visit in india",
    "places to visit in chennai",
    "best places to eat near me"
]

D = 0.75

unigram = defaultdict(int)
bigram = defaultdict(int)
trigram = defaultdict(int)
fourgram = defaultdict(int)
fivegram = defaultdict(int)

history1 = defaultdict(int)
history2 = defaultdict(int)
history3 = defaultdict(int)
history4 = defaultdict(int)

followers1 = defaultdict(set)
followers2 = defaultdict(set)
followers3 = defaultdict(set)
followers4 = defaultdict(set)

continuation = defaultdict(set)

#build model
for sentence in corpus:
    words = sentence.lower().split()
    for w in words:
        unigram[w]+=1

    #bigram
    for i in range(len(words)-1):
        w1,w2 = words[i], words[i+1]
        bigram[(w1,w2)] += 1
        history1[(w1,)] += 1
        followers1[(w1,)].add(w2)
        continuation[w2].add(w1)

    #trigram
    for i in range(len(words)-2):
        h = tuple(words[i:i+2])
        w = words[i+2]
        trigram[h+(w,)] += 1
        history2[h] += 1
        followers2[h].add(w)

    #fourgram
    for i in range(len(words)-3):
        h = tuple(words[i:i+3])
        w = words[i+3]
        fourgram[h+(w,)] += 1
        history3[h] += 1
        followers3[h].add(w)

    #fivegram
    for i in range(len(words)-4):
        h = tuple(words[i:i+4])
        w = words[i+4]
        fivegram[h+(w,)] += 1
        history4[h] += 1
        followers4[h].add(w)

total_unigrams = sum(unigram.values())
total_unique_bigrams = len(bigram)

def KN(history, word):
    n = len(history)
    if n == 0:
        return len(continuation[word]) / total_unique_bigrams
        
    elif n == 1:
        count = bigram.get(history+(word,),0)
        hist = history1.get(history,0)
        if hist == 0:
            return KN((),word)
        first = max(count-D,0)/hist
        lamb = D*len(followers1[history])/hist
        return first + lamb*KN((),word)
        
    elif n == 2:
        count = trigram.get(history+(word,),0)
        hist = history2.get(history,0)
        if hist == 0:
            return KN(history[1:],word)
        first = max(count-D,0)/hist
        lamb = D*len(followers2[history])/hist
        return first + lamb*KN(history[1:],word)
        
    elif n == 3:
        count = fourgram.get(history+(word,),0)
        hist = history3.get(history,0)
        if hist == 0:
            return KN(history[1:],word)
        first = max(count-D,0)/hist
        lamb = D*len(followers3[history])/hist
        return first + lamb*KN(history[1:],word)

    elif n == 4:
        count = fivegram.get(history+(word,),0)
        hist = history4.get(history,0)
        if hist == 0:
            return KN(history[1:],word)
        first = max(count-D,0)/hist
        lamb = D*len(followers4[history])/hist
        return first + lamb*KN(history[1:],word)

history = ("best","places","to","visit")
suggestions = [
    "in",
    "near",
    "during",
    "today",
    "kerala",
    "chennai"
]

print("History :", " ".join(history))
print()

for word in suggestions:
    print(word, ":", round(KN(history,word),4))
